# 06 — Combine Feature Stack

Load all individual feature layers from S3, apply preprocessing
(aspect sin/cos encoding, curvature smoothing), and export a single
combined Zarr store for downstream ML notebooks.

Output: `s3://.../processed/feature_stack/features_30m.zarr`

### Load all features

In [1]:
import config
import numpy as np
import xarray as xr
import rioxarray  # registers .rio accessor
from s3_utils import load_zarr, save_zarr

features = {}

# Terrain
terrain = load_zarr(config.TERRAIN_ZARR)
for var in terrain.data_vars:
    if var != 'spatial_ref':
        features[var] = terrain[var]

# Land-cover fractions
lc = load_zarr(config.LC_FRACTIONS_ZARR, name='lc_fraction')
for band in lc.band.values:
    features[f'lc_{band.lower().replace(" ", "_")}'] = lc.sel(band=band)

# CHELSA bioclimatic
for key, (path, var) in {
    'bio1': (config.CHELSA_BIO1_ZARR, 'bio1_annual_mean_temp'),
    'bio4': (config.CHELSA_BIO4_ZARR, 'bio4_temp_seasonality'),
    'bio12': (config.CHELSA_BIO12_ZARR, 'bio12_annual_precipitation_sum'),
    'bio15': (config.CHELSA_BIO15_ZARR, 'bio15_precip_seasonality'),
    'bio18': (config.CHELSA_BIO18_ZARR, 'bio18_warmest_quarter_precip'),
}.items():
    features[key] = load_zarr(path, name=var)

# Biological / anthropogenic
features['canopy_height'] = load_zarr(config.CANOPY_HEIGHT_ZARR, name='canopy_height')
features['ndvi_max'] = load_zarr(config.NDVI_ZARR, name='ndvi_max')
features['human_footprint'] = load_zarr(config.HFP_ZARR, name='human_footprint')
features['forest_edge_dist'] = load_zarr(config.FOREST_EDGE_ZARR, name='forest_edge_distance')

print(f'Loaded {len(features)} raw features')


Loaded 20 raw features


### Preprocess

In [2]:
CRS = 'EPSG:4326'

# 1. Aspect → sin/cos (handles circularity: 1° ≈ 359°)
if 'aspect' in features:
    asp = features['aspect']
    asp_rad = np.radians(asp.values)
    features['aspect_cos'] = xr.DataArray(
        np.cos(asp_rad), coords=asp.coords, dims=asp.dims, name='aspect_cos'
    ).rio.write_crs(CRS)
    features['aspect_sin'] = xr.DataArray(
        np.sin(asp_rad), coords=asp.coords, dims=asp.dims, name='aspect_sin'
    ).rio.write_crs(CRS)
    del features['aspect']
    print('Aspect → sin/cos')

# 2. Smooth curvatures (5×5 rolling mean ≈ 150 m)
for name in ['plan_curvature', 'profile_curvature']:
    if name in features:
        da = features[name]
        features[name] = da.rolling(x=5, y=5, center=True, min_periods=1).mean()
        print(f'Smoothed {name}')

print(f'\n{len(features)} features after preprocessing: {list(features.keys())}')


Aspect → sin/cos
Smoothed plan_curvature
Smoothed profile_curvature

21 features after preprocessing: ['plan_curvature', 'profile_curvature', 'slope', 'twi', 'lc_tree_cover', 'lc_grassland', 'lc_cropland', 'lc_built-up', 'lc_snow_and_ice', 'lc_permanent_water_bodies', 'bio1', 'bio4', 'bio12', 'bio15', 'bio18', 'canopy_height', 'ndvi_max', 'human_footprint', 'forest_edge_dist', 'aspect_cos', 'aspect_sin']


### Export combined feature stack

In [3]:
# Drop stale 'band' coords before merging
clean = {}
for name, da in features.items():
    if 'band' in da.coords and 'band' not in da.dims:
        da = da.drop_vars('band')
    clean[name] = da

ds = xr.Dataset(clean)
print(f'Dataset: {dict(ds.sizes)}')
print(f'Variables: {list(ds.data_vars)}')

save_zarr(ds, config.FEATURE_STACK_ZARR)


Dataset: {'x': 17810, 'y': 9010}
Variables: ['plan_curvature', 'profile_curvature', 'slope', 'twi', 'lc_tree_cover', 'lc_grassland', 'lc_cropland', 'lc_built-up', 'lc_snow_and_ice', 'lc_permanent_water_bodies', 'bio1', 'bio4', 'bio12', 'bio15', 'bio18', 'canopy_height', 'ndvi_max', 'human_footprint', 'forest_edge_dist', 'aspect_cos', 'aspect_sin']


/home/sagemaker-user/.conda/envs/cas/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Saved → s3://km-cas-datalake/processed/feature_stack/features_30m.zarr


### Export training subsample

Precompute the valid pixel mask, random subsample, and scaled training
matrix so the SOM notebook can skip straight to training.

In [4]:
import gc
from sklearn.preprocessing import RobustScaler

feature_names = [v for v in ds.data_vars if v != 'spatial_ref']
n_features = len(feature_names)
ref = ds[feature_names[0]]
grid_shape = ref.shape
n_pixels = grid_shape[0] * grid_shape[1]

# Valid mask
print('Building valid mask...')
valid_mask = np.ones(n_pixels, dtype=bool)
for name in feature_names:
    valid_mask &= np.isfinite(ds[name].values.ravel())
valid_indices = np.where(valid_mask)[0]
n_valid = len(valid_indices)
print(f'Valid: {n_valid:,} / {n_pixels:,} ({n_valid/n_pixels*100:.1f}%)')

# Subsample
N_TRAIN = 500_000
rng = np.random.default_rng(42)
train_idx = rng.choice(valid_indices, size=min(N_TRAIN, n_valid), replace=False)
train_idx.sort()

X_train = np.empty((len(train_idx), n_features), dtype=np.float32)
for j, name in enumerate(feature_names):
    X_train[:, j] = ds[name].values.ravel()[train_idx]

# Scale
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_train)
del X_train

# Post-process
def post_scale(X):
    X = X.copy()
    if 'forest_edge_dist' in feature_names:
        X[:, feature_names.index('forest_edge_dist')] = np.clip(
            X[:, feature_names.index('forest_edge_dist')], -5, 5)
    for name in feature_names:
        if name.startswith('lc_'):
            X[:, feature_names.index(name)] *= 3.0
    return np.clip(X, -5, 5)

X_scaled = post_scale(X_scaled)

# Save everything the SOM notebook needs
import s3fs as _s3fs
s3 = _s3fs.S3FileSystem(anon=False)
som_prep_path = config.S3_PROCESSED + '/som_prep'

# Training matrix as numpy
np.save('/tmp/X_train_scaled.npy', X_scaled)
s3.put('/tmp/X_train_scaled.npy', f'{som_prep_path}/X_train_scaled.npy')

# Valid indices
np.save('/tmp/valid_indices.npy', valid_indices)
s3.put('/tmp/valid_indices.npy', f'{som_prep_path}/valid_indices.npy')

# Scaler params (for applying to full dataset during BMU assignment)
import pickle
with open('/tmp/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
s3.put('/tmp/scaler.pkl', f'{som_prep_path}/scaler.pkl')

# Feature names
import json as _json
with open('/tmp/feature_names.json', 'w') as f:
    _json.dump(feature_names, f)
s3.put('/tmp/feature_names.json', f'{som_prep_path}/feature_names.json')

# Grid metadata
meta = {'grid_shape': list(grid_shape), 'n_pixels': n_pixels, 'n_valid': n_valid}
with open('/tmp/grid_meta.json', 'w') as f:
    _json.dump(meta, f)
s3.put('/tmp/grid_meta.json', f'{som_prep_path}/grid_meta.json')

print(f'Saved to {som_prep_path}/')
print(f'  X_train_scaled: {X_scaled.shape}')
print(f'  valid_indices: {valid_indices.shape}')
print(f'  feature_names: {feature_names}')
gc.collect()


Building valid mask...
Valid: 157,901,556 / 160,468,100 (98.4%)
Saved to s3://km-cas-datalake/processed/som_prep/
  X_train_scaled: (500000, 21)
  valid_indices: (157901556,)
  feature_names: ['plan_curvature', 'profile_curvature', 'slope', 'twi', 'lc_tree_cover', 'lc_grassland', 'lc_cropland', 'lc_built-up', 'lc_snow_and_ice', 'lc_permanent_water_bodies', 'bio1', 'bio4', 'bio12', 'bio15', 'bio18', 'canopy_height', 'ndvi_max', 'human_footprint', 'forest_edge_dist', 'aspect_cos', 'aspect_sin']


409